In [ ]:
import asyncio
import os
from dotenv import load_dotenv

from utils.api.riot.models.base.region import Region
from utils.api.riot.models.player.player_snapshot import PlayerSnapshot
from utils.api.riot.player_api_service import PlayerApiService

%load_ext autoreload
%autoreload 2

load_dotenv(override=True)

RIOT_PERSONAL_API_KEY = str(os.getenv("LOL_PERSONAL_API_KEY"))
POSTGRES_LOCAL_URL = os.getenv("POSTGRES_ADMIN_URL")

In [ ]:
import requests

url = "https://europe.api.riotgames.com/lol/match/v5/matches/EUN1_3956587229"

payload = {}
headers = {
    'X-Riot-Token': RIOT_PERSONAL_API_KEY
}

response = None
while response is None or response.status_code != 429:
    response = requests.request("GET", url, headers=headers, data=payload)
    response_json = response.json()
    headers = response.headers

response_json
headers

In [ ]:
import polars as pl

pl.DataFrame(response_json["info"]["participants"])

In [ ]:
%%sql
CREATE TABLE matches (
    match_id text PRIMARY KEY,
    data_version TEXT NOT NULL,
    game_result_state TEXT,

    game_creation TIMESTAMP,
    game_start TIMESTAMP,
    game_end TIMESTAMP,
    game_duration INT,

    game_id BIGINT,
    game_name TEXT,
    game_mode TEXT,
    game_type TEXT,
    game_version TEXT,
    queue_type INT,
    map_id INT,

    region TEXT,
    tournament_code TEXT
);

In [ ]:
import polars as pl


def parse_match(match_json: dict) -> dict:
    meta = match_json["metadata"]
    info = match_json["info"]
    return {
        "match_id": meta["matchId"],
        "data_version": meta["dataVersion"],
        "game_result_state": info.get("endOfGameResult"),
        "game_creation": info.get("gameCreation"),
        "game_start": info.get("gameStartTimestamp"),
        "game_end": info.get("gameEndTimestamp"),
        "game_duration": info.get("gameDuration"),
        "game_id": info.get("gameId"),
        "game_name": info.get("gameName"),
        "game_mode": info.get("gameMode"),
        "game_type": info.get("gameType"),
        "game_version": info.get("gameVersion"),
        "queue_type": info.get("queueId"),
        "map_id": info.get("mapId"),
        "region": info.get("platformId"),
        "tournament_code": info.get("tournamentCode"),
    }


def matches_to_df(match_jsons: list[dict]) -> pl.DataFrame:
    rows = [parse_match(m) for m in match_jsons]
    return (
        pl.DataFrame(rows)
        .with_columns(
            pl.col("game_creation").cast(pl.Datetime("ms", "UTC")),
            pl.col("game_start").cast(pl.Datetime("ms", "UTC")),
            pl.col("game_end").cast(pl.Datetime("ms", "UTC")),
            pl.col("game_duration").cast(pl.Int32),
            pl.col("queue_type").cast(pl.Int32),
            pl.col("map_id").cast(pl.Int32)
        )
    )


df = matches_to_df([response_json])
df

In [ ]:
df.write_database(
    table_name="matches",
    connection=POSTGRES_LOCAL_URL,
    if_table_exists="append",
    engine="adbc"
)

In [ ]:
import polars as pl

pl.read_database_uri(
    query="""
          SELECT *
          FROM players
          """,
    uri=POSTGRES_LOCAL_URL
)



In [ ]:
import json
from datetime import date
from dataclasses import dataclass
import polars as pl


@dataclass
class ChallengePointDto:
    level: str
    current: int
    max: int
    percentile: float


(pl.DataFrame(
    {
        "puuid": "caca",
        "snapshot_date": date.today(),
        "summoner_level": 1,
        "title": "caca",
        "profile_icon_id": 1,
        "crest_border": "ceva",
        "banner_accent": "ceva",
        "prestige_crest_border_level": 1,
        "total_points": json.dumps(ChallengePointDto(
            level="test",
            current=1,
            max=1,
            percentile=50.0,
        ).__dict__)
    }
)
.write_database(
    table_name="player_snapshots",
    connection=POSTGRES_LOCAL_URL,
    if_table_exists="append",
    engine="adbc"
)
)

In [ ]:
(pl.DataFrame(
    {
        "puuid": "caca",
        "snapshot_date": date.today(),
        "summoner_level": 1,
        "title": "caca",
        "profile_icon_id": 1,
        "crest_border": "ceva",
        "banner_accent": "ceva",
        "prestige_crest_border_level": 1,
        "total_points": ChallengePointDto(
            level="test",
            current=1,
            max=1,
            percentile=50.0,
        )
    }
)
 .with_columns(pl.col(pl.Struct).map_elements(json.dumps))
 .with_columns(
    pl.col(pl.Int64).cast(pl.Int32),
    pl.col(pl.Float64).cast(pl.Float32)
).schema)

In [ ]:
tuple(ChallengePointDto(
    level="test",
    current=1,
    max=1,
    percentile=50.0,
).__dict__.values())



In [ ]:
pl.DataFrame().write_parquet()

In [ ]:

async with AsyncRiotAPIClient(RIOT_PERSONAL_API_KEY, default_region=Region.EUNE, initial_capacity=100) as api_client:
    routines = [api_client.request(f"/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/I?page={num}") for num in range(200)]
    responses = await asyncio.gather(*routines)
    data = [entry.data for entry in responses if entry.status == 200]
    print(len(data))


In [ ]:
async with AsyncRiotAPIClient(RIOT_PERSONAL_API_KEY, default_region=Region.EUNE, initial_capacity=20) as api_client:
    routines = [api_client.request(f"/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/I?page={num}") for num in range(200)]
    responses = await asyncio.gather(*routines)
    data = [entry.data for entry in responses if entry.status == 200]
    print(len(data))


In [ ]:
import random
import asyncio
import time
import aiohttp


async def fetch_with_retry(session: aiohttp.ClientSession, url: str, counter: dict) -> dict | None:
    """Fire the request; while it's 429, honor Retry-After and retry. Return JSON on 200."""
    while True:
        async with session.request("GET", url) as response:
            if response.status == 429:
                retry_after = int(response.headers.get("Retry-After", 1))
                counter["rate_limited"] += 1
                await asyncio.sleep(retry_after + random.uniform(0, 1))
                continue

            counter["done"] += 1
            if response.status != 200:
                counter["failed"] += 1
                return None
            return await response.json()


async def monitor(counter: dict, total: int, start: float):
    """Print progress once a second until all requests are done."""
    while counter["done"] < total:
        await asyncio.sleep(1)
        elapsed = time.monotonic() - start
        print(
            f"[{elapsed:5.1f}s] done {counter['done']}/{total} "
            f"(failed {counter['failed']}, 429s so far {counter['rate_limited']})"
        )


async def speed_test():
    headers = {"X-Riot-Token": RIOT_PERSONAL_API_KEY}
    base = "https://eun1.api.riotgames.com"
    total = 200
    paths = [
        f"/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/I?page={num}"
        for num in range(total)
    ]

    counter = {"done": 0, "failed": 0, "rate_limited": 0}
    start = time.monotonic()

    async with aiohttp.ClientSession(headers=headers) as session:
        tasks = [fetch_with_retry(session, f"{base}{path}", counter) for path in paths]
        monitor_task = asyncio.create_task(monitor(counter, total, start))

        results = await asyncio.gather(*tasks)
        await monitor_task  # let it print the final tick and exit

    elapsed = time.monotonic() - start
    data = [r for r in results if r is not None]

    print(f"\ngot {len(data)}/{total} responses in {elapsed:.1f}s "
          f"({counter['rate_limited']} total 429 retries)")


await speed_test()

In [ ]:
from utils.postgres.postgres_connection import PostgresConnection

connection_wrapper = PostgresConnection(POSTGRES_LOCAL_URL)

In [ ]:
connection_wrapper.close()

In [ ]:
import polars as pl

class TestClass:
    def __init__(self, number: int):
        self.number = number

    def to_tuple(self) -> tuple:
        return (self.number,)

test_df = pl.DataFrame(
    {
        "id": [num for num in range(1000)],
        "dto_test": [TestClass(num) for num in range(0, 1000)],
    }
)

test_df.schema

In [ ]:
connection_wrapper.upsert_dataframe_dto(
    test_df,
    "test.test_table",
    ["id"]
)

In [21]:
from utils.api.riot.riot_games_api import AsyncRiotAPIClient
from utils.api.riot.player_api_service import PlayerApiService

async with AsyncRiotAPIClient(RIOT_PERSONAL_API_KEY, default_region=Region.EUNE, initial_capacity=20) as api_client:
    player_api_service = PlayerApiService(api_client, Region.EUNE)

    player_data = await player_api_service.request_by_game_name("Lukyen", "emo")
    player_snapshot = await player_api_service.get_player_snapshot(player_data.puuid)
    player_masteries = await player_api_service.get_player_masteries(player_data.puuid)
    player_challenges = await player_api_service.get_player_challenges(player_data.puuid)
    player_rank_snapshots = await player_api_service.get_player_rank_snapshots(player_data.puuid)

    pass

2026-06-09 17:09:32,395 [INFO] utils.api.riot.riot_games_api: async riot api client opened
2026-06-09 17:09:32,996 [INFO] utils.api.riot.riot_games_api: async riot api client closed


In [22]:
from utils.api.riot.models.player.player import PlayerBase
from utils.api.riot.models.player.player_snapshot import PlayerSnapshot
from utils.api.riot.models.player.mastery import Mastery
from utils.api.riot.models.player.challenge import Challenge
from utils.api.riot.models.rank.rank_snapshot import RankSnapshot

PlayerBase.to_dataframe([player_data])

puuid,game_name,tag_line,region,synced_at
str,str,str,str,datetime[μs]
"""sTVEkAgV_YBpZI20HXfrBzzxcrpJuU…","""Lukyen""","""emo""","""EUN1""",2026-06-09 14:09:32.524222


In [23]:
from utils.postgres.postgres_connection import PostgresConnection

with PostgresConnection(POSTGRES_LOCAL_URL) as postgres_connection:
    postgres_connection.upsert_dataframe_dto(PlayerBase.to_dataframe([player_data]), "players", PlayerBase.get_keys())
    postgres_connection.upsert_dataframe_dto(PlayerSnapshot.to_dataframe([player_snapshot]), "player_snapshots", PlayerSnapshot.get_keys())
    postgres_connection.upsert_dataframe_dto(Challenge.to_dataframe(player_challenges), "challenges", Challenge.get_keys())
    postgres_connection.upsert_dataframe_dto(Mastery.to_dataframe(player_masteries), "masteries", Mastery.get_keys())
    postgres_connection.upsert_dataframe_dto(RankSnapshot.to_dataframe(player_rank_snapshots), "rank_snapshots", RankSnapshot.get_keys())

2026-06-09 17:09:41,228 [INFO] utils.postgres.postgres_connection: opening postgres connection
2026-06-09 17:09:41,556 [INFO] utils.postgres.postgres_connection: upserted 1 rows into players, took 0.10190916061401367 seconds
2026-06-09 17:09:41,664 [INFO] utils.postgres.postgres_connection: upserted 1 rows into player_snapshots, took 0.1062016487121582 seconds
2026-06-09 17:09:41,857 [INFO] utils.postgres.postgres_connection: upserted 348 rows into challenges, took 0.18578195571899414 seconds
2026-06-09 17:09:42,011 [INFO] utils.postgres.postgres_connection: upserted 170 rows into masteries, took 0.14864563941955566 seconds
2026-06-09 17:09:42,119 [INFO] utils.postgres.postgres_connection: upserted 2 rows into rank_snapshots, took 0.1068413257598877 seconds
2026-06-09 17:09:42,120 [INFO] utils.postgres.postgres_connection: closing postgres connection
